# Stratified Evaluation

Aggregate precision and recall can hide structure: a model may look adequate
overall while failing on the tiny, densely packed objects that dominate aerial
imagery. This notebook demonstrates `evaluate_by_stratum()` from
`src/aeronetra/evaluation/strata.py`, which reports the same operating point
broken down by **object size** and **image density**.

It runs on a small synthetic example, so it needs no dataset and no weights.
See [`docs/EVALUATION.md`](../docs/EVALUATION.md) for the metrics themselves.

In [ ]:
import sys
from pathlib import Path

# Resolve the repository root regardless of where the notebook is launched,
# then expose the package source on the import path.
REPO_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
    Path.cwd(),
)
sys.path.insert(0, str(REPO_ROOT / "src"))

In [ ]:
from aeronetra.detection.types import BoundingBox, Detection, ModelPrediction
from aeronetra.evaluation import evaluate_by_stratum
from aeronetra.evaluation.types import GroundTruth, GroundTruthObject

# Two images. Image "a" holds a small car and a large car; image "b" a small car.
ground_truth = GroundTruth()
ground_truth.add_image(
    "a",
    [
        GroundTruthObject(0, BoundingBox(10, 10, 20, 20)),  # small
        GroundTruthObject(0, BoundingBox(0, 0, 200, 200)),  # large
    ],
)
ground_truth.add_image("b", [GroundTruthObject(0, BoundingBox(5, 5, 15, 15))])


def detection(x1, y1, x2, y2, confidence=0.9):
    return Detection(BoundingBox(x1, y1, x2, y2), 0, "car", confidence)


# The small car in "a" is found, the large one is missed, and there is one
# small false positive. Image "b" is detected correctly.
predictions = {
    "a": ModelPrediction([detection(10, 10, 20, 20), detection(500, 500, 510, 510)]),
    "b": ModelPrediction([detection(5, 5, 15, 15)]),
}

report = evaluate_by_stratum(predictions, ground_truth)

for title, strata in (("by size", report.by_size), ("by density", report.by_density)):
    print(title)
    for label, metrics in strata.items():
        print(
            f"  {label:11s} gt={metrics.num_ground_truth:2d} "
            f"tp={metrics.true_positives} fp={metrics.false_positives} "
            f"fn={metrics.false_negatives} P={metrics.precision:.2f} "
            f"R={metrics.recall:.2f} F1={metrics.f1:.2f}"
        )